In [88]:
import pandas as pd
import requests
import numpy as np
import openpyxl
import matplotlib.pyplot as plt
import seaborn as sns

In [89]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [90]:
#Genere un archivo txt con datos relevantes. Abro el archivo, tomo los datos necesarios y luego cierro el archivo
archivo = open('C:\\Users\\Usuario\\Desktop\\Henry-Data science\\Base de datos Yelp.txt', encoding='utf-8')
info = [dato.strip() for dato in archivo]
archivo.close()

In [91]:
#url de la API
api_url='https://api.yelp.com/v3/businesses/search'

#estos datos corresponden a una cuenta de usuario creada previamente
clientid = info[0]
apikey = info[1]
headers={'Authorization':'Bearer %s'%apikey}

In [92]:
ciudad = "Miami" # Se puede elegir cualquier ciudad del dataset entregado

In [93]:
# De esta manera se consulta el API de Yelp para obtener información sobre restaurantes en la ciudad especificada. Voy guardando en "resultados" las 6 vueltas de 40 datos
resultados = []
for i in range(0,240,40):
    params={'term':'restaurants','location': ciudad,'limit':40, 'offset':i}
    response=requests.get(api_url,params=params,headers=headers)
    data=response.json()
    resultados.extend(data['businesses'])
    print(len(resultados))
    print(data)

40
{'businesses': [{'id': 'UXHxLN3DcDGI57uDIfCuJA', 'alias': 'olds-havana-cuban-bar-and-cocina-miami', 'name': "Old's Havana Cuban Bar & Cocina", 'image_url': 'https://s3-media0.fl.yelpcdn.com/bphoto/OyMD-xvBjobfDmDdBs2Jfw/o.jpg', 'is_closed': False, 'url': 'https://www.yelp.com/biz/olds-havana-cuban-bar-and-cocina-miami?adjust_creative=GWOCZh9-BmZxtdsAjr7Gug&utm_campaign=yelp_api_v3&utm_medium=api_v3_business_search&utm_source=GWOCZh9-BmZxtdsAjr7Gug', 'review_count': 3299, 'categories': [{'alias': 'cuban', 'title': 'Cuban'}, {'alias': 'bars', 'title': 'Bars'}, {'alias': 'venues', 'title': 'Venues & Event Spaces'}], 'rating': 4.4, 'coordinates': {'latitude': 25.765607074496565, 'longitude': -80.21921771559332}, 'transactions': ['pickup'], 'price': '$$', 'location': {'address1': '1442 SW 8th St', 'address2': '', 'address3': '', 'city': 'Miami', 'zip_code': '33135', 'country': 'US', 'state': 'FL', 'display_address': ['1442 SW 8th St', 'Miami, FL 33135']}, 'phone': '+17865182196', 'displa

In [94]:
#json_normalize abre los diccionarios anidados en columnas separadas (location.city, coordinates.latitude, etc.), y cada restaurante queda en su propia fila.
df_yelp = pd.json_normalize(resultados)
df_yelp.shape            # (240, n_columnas)

(240, 24)

In [95]:
#Para dejar la columna categories legible (viene como lista de diccionarios):
df_yelp['categorias'] = df_yelp['categories'].apply(lambda cats: ', '.join(c['title'] for c in cats))

In [96]:
#Para quedarte solo con las columnas más útiles:
df_yelp = df_yelp[['id', 'name', 'rating', 'review_count', 'price', 'categorias',
                   'location.city', 'location.zip_code', 'coordinates.latitude',
                   'coordinates.longitude', 'display_phone']]
df_yelp

,id,name,rating,review_count,price,categorias,location.city,location.zip_code,coordinates.latitude,coordinates.longitude,display_phone
0,UXHxLN3DcDGI57uDIfCuJA,Old's Havana Cuban Bar & Cocina,4.4,3299,$$,"Cuban, Bars, Venues & Event Spaces",Miami,33135,25.765607,-80.219218,(786) 518-2196
1,oxtMfBGmVNE18pFVuw7lFg,Fratellino,4.8,1944,$$$,"Italian, Italian",Coral Gables,33134,25.749170,-80.260110,(786) 452-0068
2,O4gh_gZPp1zW9QtC1aMHWw,House of Food Porn,4.8,183,NaN,"Sushi Bars, Asian Fusion, Wine Bars, Sushi Bar...",Miami,33150,25.832632,-80.200107,(305) 788-0137
3,K3ukx2e11xTRtYBU01dmrA,Salty Flame,4.2,283,NaN,"Asian Fusion, Steakhouses, Asian Fusion, Steak...",Miami,33131,25.760220,-80.192670,(305) 563-8972
4,ix8ifP1jQM9ektdVAs19sQ,Crazy About You,4.2,2923,$$$,"Mediterranean, Italian, Spanish",Miami,33131,25.762015,-80.188986,(305) 377-4442
5,_tiVacKyfWZjOljFenLwVQ,Giselle Miami,4.6,533,NaN,"Mediterranean, Lounges, Asian Fusion, Mediterr...",Miami,33132,25.784964,-80.193414,(305) 358-9848
6,WqIBKxRUPXH3NqTViAHLxQ,Bayshore Club Bar & Grill,4.3,1083,$$$,"Seafood, American, Wine Bars",Miami,33133,25.728774,-80.234162,(305) 209-1200
7,scZfLg-ujWMEOUmFM_Qalg,Don Maguey Mexican Restaurant,4.7,328,$$,Mexican,Miami,33134,25.771768,-80.272538,(786) 558-5650
8,eVhl0Yu1ZRWRD2D1OQGsYg,Capriccio,4.5,193,NaN,"Italian, Pizza, Steakhouses",Miami,33126,25.781633,-80.287662,(786) 936-5470
9,79IaCcuoIGLQ-5jDMWjJLA,Dumplings Mi Amor,4.5,144,NaN,"Dim Sum, Pan Asian",Miami,33131,25.773030,-80.190122,


In [97]:
#Chequeos recomendados:
df_yelp['id'].duplicated().sum()   # Yelp a veces repite negocios entre páginas
df_yelp = df_yelp.drop_duplicates(subset='id').reset_index(drop=True)
df_yelp

,id,name,rating,review_count,price,categorias,location.city,location.zip_code,coordinates.latitude,coordinates.longitude,display_phone
0,UXHxLN3DcDGI57uDIfCuJA,Old's Havana Cuban Bar & Cocina,4.4,3299,$$,"Cuban, Bars, Venues & Event Spaces",Miami,33135,25.765607,-80.219218,(786) 518-2196
1,oxtMfBGmVNE18pFVuw7lFg,Fratellino,4.8,1944,$$$,"Italian, Italian",Coral Gables,33134,25.749170,-80.260110,(786) 452-0068
2,O4gh_gZPp1zW9QtC1aMHWw,House of Food Porn,4.8,183,NaN,"Sushi Bars, Asian Fusion, Wine Bars, Sushi Bar...",Miami,33150,25.832632,-80.200107,(305) 788-0137
3,K3ukx2e11xTRtYBU01dmrA,Salty Flame,4.2,283,NaN,"Asian Fusion, Steakhouses, Asian Fusion, Steak...",Miami,33131,25.760220,-80.192670,(305) 563-8972
4,ix8ifP1jQM9ektdVAs19sQ,Crazy About You,4.2,2923,$$$,"Mediterranean, Italian, Spanish",Miami,33131,25.762015,-80.188986,(305) 377-4442
5,_tiVacKyfWZjOljFenLwVQ,Giselle Miami,4.6,533,NaN,"Mediterranean, Lounges, Asian Fusion, Mediterr...",Miami,33132,25.784964,-80.193414,(305) 358-9848
6,WqIBKxRUPXH3NqTViAHLxQ,Bayshore Club Bar & Grill,4.3,1083,$$$,"Seafood, American, Wine Bars",Miami,33133,25.728774,-80.234162,(305) 209-1200
7,scZfLg-ujWMEOUmFM_Qalg,Don Maguey Mexican Restaurant,4.7,328,$$,Mexican,Miami,33134,25.771768,-80.272538,(786) 558-5650
8,eVhl0Yu1ZRWRD2D1OQGsYg,Capriccio,4.5,193,NaN,"Italian, Pizza, Steakhouses",Miami,33126,25.781633,-80.287662,(786) 936-5470
9,79IaCcuoIGLQ-5jDMWjJLA,Dumplings Mi Amor,4.5,144,NaN,"Dim Sum, Pan Asian",Miami,33131,25.773030,-80.190122,


In [98]:
#Veamos los keys del diccionario recibido
data.keys()

dict_keys(['businesses', 'total', 'region'])

In [99]:
#El primer elemento del diccionario indica el total de restaurants existentes en la API
print('En total la base de datos registra %d restaurants'%data['total'])

En total la base de datos registra 5900 restaurants


In [ ]:
data['businesses'] # Hasta acá se le entregaría a los estudiantes

In [100]:
#Las filas que no tienen precio desido borrarlas
df_yelp = df_yelp.dropna(subset=['price']).reset_index(drop=True)
df_yelp

,id,name,rating,review_count,price,categorias,location.city,location.zip_code,coordinates.latitude,coordinates.longitude,display_phone
0,UXHxLN3DcDGI57uDIfCuJA,Old's Havana Cuban Bar & Cocina,4.4,3299,$$,"Cuban, Bars, Venues & Event Spaces",Miami,33135,25.765607,-80.219218,(786) 518-2196
1,oxtMfBGmVNE18pFVuw7lFg,Fratellino,4.8,1944,$$$,"Italian, Italian",Coral Gables,33134,25.749170,-80.260110,(786) 452-0068
2,ix8ifP1jQM9ektdVAs19sQ,Crazy About You,4.2,2923,$$$,"Mediterranean, Italian, Spanish",Miami,33131,25.762015,-80.188986,(305) 377-4442
3,WqIBKxRUPXH3NqTViAHLxQ,Bayshore Club Bar & Grill,4.3,1083,$$$,"Seafood, American, Wine Bars",Miami,33133,25.728774,-80.234162,(305) 209-1200
4,scZfLg-ujWMEOUmFM_Qalg,Don Maguey Mexican Restaurant,4.7,328,$$,Mexican,Miami,33134,25.771768,-80.272538,(786) 558-5650
5,hZm7TunlrksQbgS0ssXbUg,Versailles,4.0,7098,$$,"Cuban, Coffee & Tea, Sandwiches",Miami,33135,25.765390,-80.252874,(305) 444-0240
6,qnCsyyEK01rRdrZgqqSfPA,TastiGo Kitchen,4.4,81,$$,"Chinese, Noodles",Miami,33133,25.740948,-80.238151,(305) 530-8315
7,V3btCvWa1flTBYt2YKea_g,Bombay Darbar,4.5,1723,$$,"Indian, Indian",Miami,33133,25.729350,-80.240789,(305) 444-7272
8,QQnw1bFxLyamuxn4giIy9g,Sufrat Mediterranean Grill,4.4,175,$$,"Mediterranean, Greek, Lebanese",Doral,33166,25.820434,-80.335699,(305) 798-7278
9,LPVv4wk9gwGlIEdfZJRaHQ,Jrk!,4.5,154,$$,Caribbean,Miami,33128,25.775206,-80.194967,(305) 290-4684


In [101]:
#Queda un único DataFrame, df_seleccion, con los restaurantes que cumplen el criterio de cada nivel de precio. El reset_index(drop=True) renumera las filas desde 0.
# Grupo 1: $$ con rating > 4.4 y reviews > 974
top_dos = df_yelp[(df_yelp['price'] == '$$') &
                  (df_yelp['rating'] > 4.4) &
                  (df_yelp['review_count'] > 974)]

# Grupo 2: $$$ y $$$$ con rating > 4.3 y reviews > 427
top_caros = df_yelp[(df_yelp['price'].isin(['$$$', '$$$$'])) &
                    (df_yelp['rating'] > 4.3) &
                    (df_yelp['review_count'] > 427)]

# Unir en un solo DataFrame
df_seleccion = pd.concat([top_dos, top_caros]).reset_index(drop=True)

df_seleccion

,id,name,rating,review_count,price,categorias,location.city,location.zip_code,coordinates.latitude,coordinates.longitude,display_phone
0,V3btCvWa1flTBYt2YKea_g,Bombay Darbar,4.5,1723,$$,"Indian, Indian",Miami,33133,25.729350,-80.240789,(305) 444-7272
1,9h9nMMvnINJWMt7TjJtePw,The Taco Stand,4.5,1764,$$,Mexican,Miami,33127,25.800782,-80.201316,(786) 580-4948
2,fMk82DdXMJ_MULphyQM74w,La Fresa Francesa,4.6,1051,$$,"Wine Bars, French, Wine Bars, French, Wine Bar...",Hialeah,33010,25.824568,-80.282478,(786) 717-6886
3,oxtMfBGmVNE18pFVuw7lFg,Fratellino,4.8,1944,$$$,"Italian, Italian",Coral Gables,33134,25.749170,-80.260110,(786) 452-0068
4,0EVWp_KyFdWC9VxJ7Sku1Q,COTE Miami,4.5,1054,$$$$,"Korean, Steakhouses, Korean, Steakhouses, Kore...",Miami,33137,25.812906,-80.191491,(305) 434-4668
5,M8dlhbSz62XnqJ4mP64L4w,Luca Osteria,4.5,510,$$$,Italian,Coral Gables,33134,25.751177,-80.257179,(305) 381-5097
6,MfkZSdJ0w4-SOLJiEyHU5A,Kyu,4.4,2133,$$$,Asian Fusion,Miami,33127,25.800890,-80.200170,(786) 577-0150
7,1h8-r9cLeXxW99HhlRUnlA,CVI.CHE 105,4.4,6227,$$$,"Peruvian, Seafood",Miami,33132,25.775383,-80.188400,(305) 577-3454
8,Obt7Njta79-1CNtNwhNXiw,Bunbury,4.6,986,$$$,"Argentine, Steakhouses, Cocktail Bars, Argenti...",Miami,33132,25.788458,-80.193363,(305) 333-6929
9,nTsEr_CQqsA8zwLV4kT5nA,Cafe La Trova,4.4,1337,$$$,"Cocktail Bars, Cuban, Venues & Event Spaces, C...",Miami,33130,25.766266,-80.210630,(786) 615-4379
